<a href="https://colab.research.google.com/github/NhatChaMuc/CLONE/blob/main/NCKH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NEW TRAINING 1

In [ ]:
# ==========================================
# 🚀 TRAIN MỚI TỪ ĐẦU - MAIL CHÍNH
# ==========================================
!pip install -q -U ultralytics
import os, random, shutil, yaml
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

# 1. Khai báo đường dẫn
ZIP_PATH = '/content/drive/MyDrive/SafeTruckVision.zip'
PROJECT_PATH = '/content/drive/MyDrive/NCKH_v26_Product'
BASE_PATH = '/content/NCKH/data/processed'

train_img = os.path.join(BASE_PATH, 'images/train')
train_lbl = os.path.join(BASE_PATH, 'labels/train')
val_img = os.path.join(BASE_PATH, 'images/val')
val_lbl = os.path.join(BASE_PATH, 'labels/val')

# 2. Giải nén dữ liệu mới nhất
if os.path.exists('/content/NCKH'): shutil.rmtree('/content/NCKH')
print("📦 Đang giải nén dữ liệu...")
!unzip -q -o {ZIP_PATH} -d /content/

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

# 3. Chia tập VAL chuẩn (20%)
print("🗂️ Đang chia tập VAL...")
imgs = [f for f in os.listdir(train_img) if f.endswith(('.jpg', '.png'))]
num_val = int(len(imgs) * 0.2)
for img in random.sample(imgs, num_val):
    shutil.move(os.path.join(train_img, img), os.path.join(val_img, img))
    lbl = os.path.splitext(img)[0] + '.txt'
    if os.path.exists(os.path.join(train_lbl, lbl)):
        shutil.move(os.path.join(train_lbl, lbl), os.path.join(val_lbl, lbl))

# 4. NHÂN BẢN CỰC MẠNH (Fix lỗi nhầm Car/Truck trên cao tốc)
# Tăng cường lớp 2 (Car), 5 (Truck) để đối trọng với Xe máy (3)
target_classes = ['2', '5']
print("⚖️ Đang nhân bản x10 cho Car và Truck để cân bằng dữ liệu...")
for file in [f for f in os.listdir(train_lbl) if f.endswith('.txt')]:
    with open(os.path.join(train_lbl, file), 'r') as f:
        content = f.read()
        if any(cls in content for cls in target_classes):
            img_ext = '.jpg' if os.path.exists(os.path.join(train_img, file.replace('.txt', '.jpg'))) else '.png'
            img_file = file.replace('.txt', img_ext)
            for i in range(10): # Nhân bản gấp 10 lần
                shutil.copy(os.path.join(train_img, img_file), os.path.join(train_img, f"aug_new_{i}_{img_file}"))
                shutil.copy(os.path.join(train_lbl, file), os.path.join(train_lbl, f"aug_new_{i}_{file}"))

# 5. Cấu hình YAML (0-5 liên tục)
data_yaml = {'path': BASE_PATH, 'train': 'images/train', 'val': 'images/val', 'nc': 6,
             'names': {0: 'NGUOI DI BO', 1: 'XE DAP', 2: 'O TO', 3: 'XE MAY', 4: 'XE BUYT', 5: 'XE TAI'}}
with open(os.path.join(BASE_PATH, 'data.yaml'), 'w') as f:
    yaml.dump(data_yaml, f)

# 6. Train mới hoàn toàn
print("🚀 Bắt đầu huấn luyện bản Stable Final...")
model = YOLO('yolo26n.pt') # Load pre-trained gốc
model.train(
    data=os.path.join(BASE_PATH, 'data.yaml'),
    epochs=150,
    imgsz=640,
    batch=32,
    project=PROJECT_PATH,
    name='SafeTruck_v26n_Stable_Final',
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    close_mosaic=20,
    exist_ok=True
)

NEW TRAINING 2

In [ ]:
# ==========================================
# 🚀 TRAIN MỚI TỪ ĐẦU - MAIL PHỤ (SHORTCUT)
# ==========================================
!pip install -q -U ultralytics
import os, random, shutil, yaml
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

# 1. Đường dẫn Shortcut
ZIP_PATH = '/content/drive/MyDrive/SafeTruckVision.zip'
PROJECT_PATH = '/content/drive/MyDrive/NCKH_v26_Product' # Shortcut thư mục kết quả
BASE_PATH = '/content/NCKH/data/processed'

train_img, train_lbl = os.path.join(BASE_PATH, 'images/train'), os.path.join(BASE_PATH, 'labels/train')
val_img, val_lbl = os.path.join(BASE_PATH, 'images/val'), os.path.join(BASE_PATH, 'labels/val')

# 2. Làm sạch máy ảo và giải nén
if os.path.exists('/content/NCKH'): shutil.rmtree('/content/NCKH')
!unzip -q -o {ZIP_PATH} -d /content/
os.makedirs(val_img, exist_ok=True); os.makedirs(val_lbl, exist_ok=True)

# 3. Chia VAL (20%)
imgs = [f for f in os.listdir(train_img) if f.endswith(('.jpg', '.png'))]
for img in random.sample(imgs, int(len(imgs) * 0.2)):
    shutil.move(os.path.join(train_img, img), os.path.join(val_img, img))
    lbl = os.path.splitext(img)[0] + '.txt'
    if os.path.exists(os.path.join(train_lbl, lbl)):
        shutil.move(os.path.join(train_lbl, lbl), os.path.join(val_lbl, lbl))

# 4. Nhân bản lớp Ô tô & Xe tải (x10)
for file in [f for f in os.listdir(train_lbl) if f.endswith('.txt')]:
    with open(os.path.join(train_lbl, file), 'r') as f:
        if any(cls in f.read() for cls in ['2', '5']):
            img_ext = '.jpg' if os.path.exists(os.path.join(train_img, file.replace('.txt', '.jpg'))) else '.png'
            img_f = file.replace('.txt', img_ext)
            if os.path.exists(os.path.join(train_img, img_f)):
                for i in range(10):
                    shutil.copy(os.path.join(train_img, img_f), os.path.join(train_img, f"aug_new_{i}_{img_f}"))
                    shutil.copy(os.path.join(train_lbl, file), os.path.join(train_lbl, f"aug_new_{i}_{file}"))

# 5. Cấu hình YAML
with open(os.path.join(BASE_PATH, 'data.yaml'), 'w') as f:
    yaml.dump({'path': BASE_PATH, 'train': 'images/train', 'val': 'images/val', 'nc': 6,
               'names': {0: 'NGUOI DI BO', 1: 'XE DAP', 2: 'O TO', 3: 'XE MAY', 4: 'XE BUYT', 5: 'XE TAI'}}, f)

# 6. Train mới
YOLO('yolo26n.pt').train(
    data=os.path.join(BASE_PATH, 'data.yaml'),
    epochs=150, imgsz=640, batch=32,
    project=PROJECT_PATH, name='SafeTruck_v26n_Stable_Final',
    optimizer='AdamW', lr0=0.001, cos_lr=True, close_mosaic=20, exist_ok=True
)

RESUME 1

In [ ]:
# ==========================================
# 🚀 1-CLICK RESUME CHO MAIL CHÍNH
# ==========================================
!pip install -q -U ultralytics
import os, random, shutil, yaml
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

# 1. Đường dẫn GỐC trên Mail Chính
ZIP_PATH = '/content/drive/MyDrive/SafeTruckVision.zip'
WEIGHTS_PATH = '/content/drive/MyDrive/NCKH_v26_Product/SafeTruck_v26n_Stable_Final/weights/last.pt'
BASE_PATH = '/content/NCKH/data/processed'

train_img, train_lbl = os.path.join(BASE_PATH, 'images/train'), os.path.join(BASE_PATH, 'labels/train')
val_img, val_lbl = os.path.join(BASE_PATH, 'images/val'), os.path.join(BASE_PATH, 'labels/val')

# 2. Giải nén (Bỏ qua nếu đã có sẵn)
if not os.path.exists(train_img):
    print("📦 Đang giải nén dữ liệu...")
    !unzip -q -o {ZIP_PATH} -d /content/
os.makedirs(val_img, exist_ok=True); os.makedirs(val_lbl, exist_ok=True)

# 3. Chia tập Val 20% (Bỏ qua nếu đã chia)
if len(os.listdir(val_img)) == 0:
    print("🗂️ Đang chia tập VAL...")
    imgs = [f for f in os.listdir(train_img) if f.endswith(('.jpg', '.png')) and not f.startswith('stable_aug_')]
    for img in random.sample(imgs, int(len(imgs) * 0.2)):
        shutil.move(os.path.join(train_img, img), os.path.join(val_img, img))
        lbl = os.path.splitext(img)[0] + '.txt'
        if os.path.exists(os.path.join(train_lbl, lbl)):
            shutil.move(os.path.join(train_lbl, lbl), os.path.join(val_lbl, lbl))

# 4. Oversampling Ô tô/Xe buýt/Xe tải (Bỏ qua nếu đã nhân bản)
if not any(f.startswith('stable_aug_') for f in os.listdir(train_img)):
    print("⚖️ Đang cân bằng dữ liệu...")
    for file in [f for f in os.listdir(train_lbl) if f.endswith('.txt')]:
        with open(os.path.join(train_lbl, file), 'r') as f:
            if any(c in f.read() for c in ['2', '4', '5']):
                img_file = file.replace('.txt', '.jpg') if os.path.exists(os.path.join(train_img, file.replace('.txt', '.jpg'))) else file.replace('.txt', '.png')
                if os.path.exists(os.path.join(train_img, img_file)):
                    for i in range(5):
                        shutil.copy(os.path.join(train_img, img_file), os.path.join(train_img, f"stable_aug_{i}_{img_file}"))
                        shutil.copy(os.path.join(train_lbl, file), os.path.join(train_lbl, f"stable_aug_{i}_{file}"))

# 5. Cấu hình YAML & Resume
yaml_path = os.path.join(BASE_PATH, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump({'path': BASE_PATH, 'train': 'images/train', 'val': 'images/val', 'nc': 6, 'names': {0: 'NGUOI DI BO', 1: 'XE DAP', 2: 'O TO', 3: 'XE MAY', 4: 'XE BUYT', 5: 'XE TAI'}}, f)

if os.path.exists(WEIGHTS_PATH):
    print("🚀 Bắt đầu Resume chặng cuối!")
    YOLO(WEIGHTS_PATH).train(data=yaml_path, resume=True)
else:
    print("❌ Không tìm thấy last.pt. Kiểm tra lại đường dẫn.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive
📦 Đang giải nén dữ liệu...
🗂️ Đang chia tập VAL...
⚖️ Đang cân bằng dữ liệu...
🚀 Bắt đầu Resume chặng cuối!
Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/NCKH/data/processed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0

RESUME 2

In [ ]:
# ==========================================
# 🚀 1-CLICK RESUME CHO MAIL PHỤ (SHORTCUT)
# ==========================================
!pip install -q -U ultralytics
import os, random, shutil, yaml
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

# 1. Đường dẫn SHORTCUT trên Mail Phụ
ZIP_PATH = '/content/drive/MyDrive/SafeTruckVision.zip'
WEIGHTS_PATH = '/content/drive/MyDrive/SafeTruck_v26n_Stable_Final/weights/last.pt'
BASE_PATH = '/content/NCKH/data/processed'

train_img, train_lbl = os.path.join(BASE_PATH, 'images/train'), os.path.join(BASE_PATH, 'labels/train')
val_img, val_lbl = os.path.join(BASE_PATH, 'images/val'), os.path.join(BASE_PATH, 'labels/val')

# 2. Giải nén (Bỏ qua nếu đã có sẵn trên máy ảo của mail phụ)
if not os.path.exists(train_img):
    print("📦 Đang giải nén dữ liệu từ Shortcut...")
    !unzip -q -o {ZIP_PATH} -d /content/
os.makedirs(val_img, exist_ok=True); os.makedirs(val_lbl, exist_ok=True)

# 3. Chia tập Val 20% (Bỏ qua nếu đã chia)
if len(os.listdir(val_img)) == 0:
    print("🗂️ Đang chia tập VAL...")
    imgs = [f for f in os.listdir(train_img) if f.endswith(('.jpg', '.png')) and not f.startswith('stable_aug_')]
    for img in random.sample(imgs, int(len(imgs) * 0.2)):
        shutil.move(os.path.join(train_img, img), os.path.join(val_img, img))
        lbl = os.path.splitext(img)[0] + '.txt'
        if os.path.exists(os.path.join(train_lbl, lbl)):
            shutil.move(os.path.join(train_lbl, lbl), os.path.join(val_lbl, lbl))

# 4. Oversampling Ô tô/Xe buýt/Xe tải (Bỏ qua nếu đã nhân bản)
if not any(f.startswith('stable_aug_') for f in os.listdir(train_img)):
    print("⚖️ Đang cân bằng dữ liệu...")
    for file in [f for f in os.listdir(train_lbl) if f.endswith('.txt')]:
        with open(os.path.join(train_lbl, file), 'r') as f:
            if any(c in f.read() for c in ['2', '4', '5']):
                img_file = file.replace('.txt', '.jpg') if os.path.exists(os.path.join(train_img, file.replace('.txt', '.jpg'))) else file.replace('.txt', '.png')
                if os.path.exists(os.path.join(train_img, img_file)):
                    for i in range(5):
                        shutil.copy(os.path.join(train_img, img_file), os.path.join(train_img, f"stable_aug_{i}_{img_file}"))
                        shutil.copy(os.path.join(train_lbl, file), os.path.join(train_lbl, f"stable_aug_{i}_{file}"))

# 5. Cấu hình YAML & Resume
yaml_path = os.path.join(BASE_PATH, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump({'path': BASE_PATH, 'train': 'images/train', 'val': 'images/val', 'nc': 6, 'names': {0: 'NGUOI DI BO', 1: 'XE DAP', 2: 'O TO', 3: 'XE MAY', 4: 'XE BUYT', 5: 'XE TAI'}}, f)

if os.path.exists(WEIGHTS_PATH):
    print("🚀 Bắt đầu Resume chặng cuối!")
    YOLO(WEIGHTS_PATH).train(data=yaml_path, resume=True)
else:
    print("❌ Không tìm thấy last.pt. Kiểm tra lại Shortcut.")

📦 Đang chia 20% dữ liệu gốc sang VAL...
✅ Đã chia 307 ảnh sang thư mục VAL an toàn.
⏳ Đang nhân bản dữ liệu (Oversampling) để model phân biệt rõ Car/Truck...
✅ Hoàn tất! Đã thêm 6145 mẫu. Cấu trúc dữ liệu đã cân bằng.
✅ File data.yaml đã được chốt.
🚀 Mọi thứ đã hoàn hảo! Bắt đầu Resume từ Epoch 38...
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/NCKH/data/processed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=F

TEST VIDEO


In [ ]:
from ultralytics import YOLO
import os

# 1. Trỏ vào "bộ não" của bạn
# Hãy chắc chắn đường dẫn này đúng. Nếu chưa train xong, đổi 'best.pt' thành 'last.pt'
model_path = '/content/drive/MyDrive/NCKH_v26_Product/SafeTruck_v26n_Stable_Final/weights/best.pt'

if os.path.exists(model_path):
    print("🧠 Đã nạp thành công bộ não AI!")
    model = YOLO(model_path)
else:
    print("❌ Lỗi: Không tìm thấy file trọng số. Bạn kiểm tra lại đường dẫn nhé.")

# 2. Đường dẫn video đầu vào (Bạn sửa tên 'video_test.mp4' cho khớp với file của bạn)
video_input = '/content/NCKH/videos/test3.mp4'

# 3. Chạy nhận diện
if os.path.exists(video_input):
    print("🎥 Đang soi video... Hãy kiên nhẫn một chút nhé!")

    # Các tham số cực kỳ quan trọng để video mượt mà và không bị nhiễu
    results = model.predict(
        source=video_input,
        save=True,       # Bắt buộc True để lưu ra video mới
        conf=0.7,        # Độ tự tin 40% trở lên mới vẽ khung (giảm nhận diện bậy)
        iou=0.45,        # Thuật toán IoU giúp lọc các khung hình bị chồng chéo lên nhau
        line_width=2,    # Độ dày khung viền (tùy chỉnh cho đẹp)
        project='/content/drive/MyDrive', # Lưu thẳng kết quả ra Drive cho dễ tải
        name='Video_Ket_Qua'              # Tên thư mục chứa video
    )

    print("✅ Đã xong! Bạn hãy vào Drive của mình, tìm thư mục 'Video_Ket_Qua' để tải video về xem nhé.")
else:
    print("❌ Không tìm thấy video đầu vào. Bạn đã kéo thả video vào Colab chưa?")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
❌ Lỗi: Không tìm thấy file trọng số. Bạn kiểm tra lại đường dẫn nhé.
❌ Không tìm thấy video đầu vào. Bạn đã kéo thả video vào Colab chưa?
